# Jira Dev Buddy (notebook)

 **Gradio** chat → Agents SDK (planner + web search) → Responses API with mock Jira/calendar tools.

**Setup:** `OPENAI_API_KEY` in environment or repo-root `.env`. Run cells top to bottom.

**Jupyter / Gradio:** The first code cell defines `_run_coro_sync()` so planner + search run without patching `asyncio` globally (patching breaks **uvicorn** on Python 3.12, which passes `loop_factory` to `asyncio.run`).

**Run UI:** Last cell uses `demo.launch(inline=True)`; omit `inline=True` or use `launch()` for a browser tab.

In [ ]:
from __future__ import annotations

import asyncio
import concurrent.futures
import json
import os
import re
from datetime import datetime
from typing import Any, Callable, Coroutine, TypeVar

_T = TypeVar("_T")

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

from agents import Agent, ModelSettings, Runner, WebSearchTool, trace

load_dotenv(override=True)

# Hugging Face secrets and some editors add a trailing newline to the key. HTTP headers cannot
# contain \n, which breaks Bearer auth and causes "Illegal header value" on trace uploads.
_k = (os.getenv("OPENAI_API_KEY") or "").strip()
if _k:
    os.environ["OPENAI_API_KEY"] = _k


def _run_coro_sync(factory: Callable[[], Coroutine[Any, Any, _T]]) -> _T:
    """Run async code from sync callers without nest_asyncio (avoids breaking uvicorn/Gradio on 3.12)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(factory())

    def _in_thread() -> _T:
        return asyncio.run(factory())

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_in_thread).result()


### System prompt (Responses API)

In [ ]:
SYSTEM_PROMPT = """
You are **Jira Dev Buddy**, an agentic assistant for **junior software developers**. You help them
turn a Jira task into an actionable plan: **subtasks**, **time estimates**, **calendar blocking**,
**sync meetings with clear agendas** when needed, **edge cases**, and **how to test**.

## Web research (already done before you run)

The user message includes a section **### Web research (planner + search subagents)** with live web
findings from dedicated search agents. **Treat that block as primary grounding** for technical
accuracy, estimates, edge cases, and tests. **Integrate** it into your answer; do not repeat it
verbatim as a wall of text. Hosted **web_search** is **not** available in this step — use only that
pasted research plus your reasoning.

## Tools (use them; do not invent calendar or Jira state)

1. **`fetch_jira_issue`** — When the user gives an issue key (e.g. PROJ-101), load details. If the
   tool returns `ok: false`, work from the user’s pasted title/description/acceptance criteria or
   suggest they use a demo key from the error’s `known_keys`.

2. **`list_day_calendar`** — **Always** before adding blocks or meetings for that day. Times are
   **local** `HH:MM` (24h). Respect stand-up, lunch, and existing blocks.

3. **`add_calendar_block`** — After you have a sane schedule, **create one block per subtask** (or
   grouped blocks if the user prefers), with `title` matching the subtask. If you get an overlap
   error, **choose another slot** and retry.

4. **`schedule_team_call`** — When the work needs alignment (e.g. **schema**, **migrations**,
   **transaction boundaries**, **permissions**, **data model**), schedule a short sync with the
   right team. **`agenda` is mandatory**: purpose, **decisions needed**, **questions for them**,
   **timeboxed sections**, and **expected outcomes**. Link `related_issue_key` when known.

## Planning rules

- **Decompose** the issue into ordered subtasks (dependencies first). Each subtask: clear outcome,
  **estimate in minutes or hours** (ranges ok for juniors), and **definition of done**.
- **Sum of estimates** vs **free slots**: if the day is too full, say so and propose **what fits
  today** vs **backlog for tomorrow**, or **shorter timeboxes** with explicit tradeoffs.
- **Edge cases**: null/empty inputs, idempotency, concurrency, authz, timeouts, partial failures,
  rollback, env differences (local/stage/prod), feature flags, backwards compatibility, rate limits,
  large payloads, clock skew, etc. Tie each to **how you’d test** it.
- **Testing section**: unit tests, integration points, happy path, negative cases, observability,
  smoke checks after deploy if relevant.
- If the user **does not specify a date**, ask once for **today’s date** (YYYY-MM-DD) and timezone
  assumptions, or infer “today” only if they explicitly say so — otherwise **ask**.

## Tone

Supportive, concrete, no jargon walls. Prefer bullet lists and tables where helpful.

## Reply structure (use these headings)

1. **Issue recap** — Key, summary, goal in one short paragraph.
2. **Subtasks** — Ordered list: title, estimate, depends on, definition of done.
3. **Calendar** — What you read from `list_day_calendar`, then what you **booked** (blocks + calls).
4. **Meetings** — Only if needed; each with **title**, **time**, **team**, and **agenda** (echo what
   was sent to `schedule_team_call`).
5. **Edge cases & risks** — Bullets: scenario → mitigation → how to test (ground in the research block
   where it helps).
6. **Test plan** — Grouped: automated / manual / data / non-functional.
7. **Research notes** — 3–8 bullets: what from the **pre-collected web research** most changed the plan
   (estimates, risks, tests, meetings). Keep **citations / links** from that section visible.

If tools fail, explain briefly and give a **fallback plan** (e.g. user books manually in Outlook).
"""


### Guardrails

In [ ]:
_UNSAFE_PATTERNS: list[tuple[str, str]] = [
    (r"\bguaranteed\b", "intended"),
    (r"\bwill\s+never\s+fail\b", "may still fail; plan for failure modes"),
    (r"\bprod\s+is\s+safe\b", "verify in staging; production needs your team's checks"),
]


def apply_guardrails(output: str) -> str:
    for pattern, repl in _UNSAFE_PATTERNS:
        output = re.sub(pattern, repl, output, flags=re.IGNORECASE)
    disclaimer = (
        "\n\n---\n"
        "**Note:** Calendar and Jira data here are **mocked** for learning. "
        "Time estimates and agendas are **suggestions** — confirm with your team, "
        "update real Jira/Calendar, and follow your org's engineering practices."
    )
    return output + disclaimer


### Mock Jira + calendar + tool schemas

In [ ]:
_CALENDAR: dict[str, list[dict[str, Any]]] = {}

_JIRA: dict[str, dict[str, Any]] = {
    "PROJ-101": {
        "key": "PROJ-101",
        "summary": "Add retry logic to payment webhook handler",
        "description": (
            "Payments sometimes fail transiently; we need retries with backoff and idempotency. "
            "Coordinate with the database team on transaction boundaries."
        ),
        "labels": ["backend", "payments", "reliability"],
        "issue_type": "Story",
        "status": "To Do",
    },
    "PROJ-88": {
        "key": "PROJ-88",
        "summary": "Spike: evaluate read replicas for reporting queries",
        "description": "Short spike; document tradeoffs and rough cost. No prod changes yet.",
        "labels": ["database", "spike"],
        "issue_type": "Spike",
        "status": "In Progress",
    },
}


def _today_default_events(date: str) -> list[dict[str, Any]]:
    return [
        {
            "id": "seed-1",
            "start_time": "09:00",
            "end_time": "09:30",
            "title": "Team stand-up",
            "type": "meeting",
            "source": "default",
        },
        {
            "id": "seed-2",
            "start_time": "12:00",
            "end_time": "13:00",
            "title": "Lunch",
            "type": "block",
            "source": "default",
        },
    ]


def _parse_hhmm(s: str) -> tuple[int, int]:
    m = re.match(r"^(\d{1,2}):(\d{2})$", (s or "").strip())
    if not m:
        raise ValueError(f"Invalid time (use HH:MM 24h): {s!r}")
    h, mm = int(m.group(1)), int(m.group(2))
    if not (0 <= h <= 23 and 0 <= mm <= 59):
        raise ValueError(f"Invalid time: {s!r}")
    return h, mm


def _minutes(t: str) -> int:
    h, mm = _parse_hhmm(t)
    return h * 60 + mm


def _overlaps(a_start: str, a_end: str, b_start: str, b_end: str) -> bool:
    return _minutes(a_start) < _minutes(b_end) and _minutes(b_start) < _minutes(a_end)


def fetch_jira_issue(issue_key: str) -> dict[str, Any]:
    key = (issue_key or "").strip().upper()
    if not key:
        return {"ok": False, "error": "issue_key is required"}
    if key not in _JIRA:
        return {
            "ok": False,
            "error": f"No mock issue {key!r}. Try PROJ-101 or PROJ-88, or work from the user's pasted description.",
            "known_keys": list(_JIRA.keys()),
        }
    return {"ok": True, "issue": _JIRA[key]}


def list_day_calendar(date: str) -> dict[str, Any]:
    raw = (date or "").strip()
    try:
        datetime.strptime(raw, "%Y-%m-%d")
    except ValueError:
        return {"ok": False, "error": "date must be YYYY-MM-DD"}

    custom = list(_CALENDAR.get(raw, []))
    events = custom if custom else _today_default_events(raw)

    return {
        "ok": True,
        "date": raw,
        "events": sorted(events, key=lambda e: _minutes(e["start_time"])),
        "note": "Mock calendar; production would use Google Calendar / Outlook APIs.",
    }


def add_calendar_block(
    date: str,
    start_time: str,
    end_time: str,
    title: str,
    notes: str = "",
) -> dict[str, Any]:
    raw = (date or "").strip()
    try:
        datetime.strptime(raw, "%Y-%m-%d")
    except ValueError:
        return {"ok": False, "error": "date must be YYYY-MM-DD"}

    try:
        if _minutes(start_time) >= _minutes(end_time):
            return {"ok": False, "error": "start_time must be before end_time"}
    except ValueError as e:
        return {"ok": False, "error": str(e)}

    if _CALENDAR.setdefault(raw, []):
        pool = _CALENDAR[raw]
    else:
        pool = _today_default_events(raw)
        _CALENDAR[raw] = list(pool)

    for ev in pool:
        if _overlaps(start_time, end_time, ev["start_time"], ev["end_time"]):
            return {"ok": False, "error": "Overlap with existing event", "conflict": ev}

    entry = {
        "id": f"blk-{len(pool) + 1}",
        "start_time": start_time,
        "end_time": end_time,
        "title": (title or "").strip() or "Focus block",
        "type": "focus",
        "notes": (notes or "").strip(),
        "source": "user_session",
    }
    pool.append(entry)
    _CALENDAR[raw] = sorted(pool, key=lambda e: _minutes(e["start_time"]))
    return {"ok": True, "event": entry, "date": raw}


def schedule_team_call(
    date: str,
    start_time: str,
    end_time: str,
    team: str,
    meeting_title: str,
    agenda: str,
    related_issue_key: str = "",
) -> dict[str, Any]:
    raw = (date or "").strip()
    try:
        datetime.strptime(raw, "%Y-%m-%d")
    except ValueError:
        return {"ok": False, "error": "date must be YYYY-MM-DD"}

    try:
        if _minutes(start_time) >= _minutes(end_time):
            return {"ok": False, "error": "start_time must be before end_time"}
    except ValueError as e:
        return {"ok": False, "error": str(e)}

    if not (agenda or "").strip():
        return {"ok": False, "error": "agenda is required (decisions, questions, timeboxed items)"}

    pool = _CALENDAR.setdefault(raw, [])
    if not pool:
        pool.extend(_today_default_events(raw))

    for ev in pool:
        if _overlaps(start_time, end_time, ev["start_time"], ev["end_time"]):
            return {
                "ok": False,
                "error": "Overlap with existing event; pick another slot",
                "conflict": ev,
            }

    entry = {
        "id": f"call-{len(pool) + 1}",
        "start_time": start_time,
        "end_time": end_time,
        "title": (meeting_title or "").strip() or f"Sync: {team}",
        "type": "meeting",
        "team": (team or "").strip(),
        "agenda": agenda.strip(),
        "related_issue_key": (related_issue_key or "").strip().upper(),
        "source": "user_session",
    }
    pool.append(entry)
    _CALENDAR[raw] = sorted(pool, key=lambda e: _minutes(e["start_time"]))
    return {"ok": True, "event": entry, "date": raw}


_FUNCTION_TOOL_SCHEMAS: list[dict[str, Any]] = [
    {
        "type": "function",
        "name": "fetch_jira_issue",
        "description": (
            "Load mock Jira issue details by key (e.g. PROJ-101). "
            "If unknown, ask the user to paste summary/AC or use a known demo key."
        ),
        "parameters": {
            "type": "object",
            "properties": {"issue_key": {"type": "string", "description": "Jira issue key"}},
            "required": ["issue_key"],
        },
    },
    {
        "type": "function",
        "name": "list_day_calendar",
        "description": (
            "Get that day's calendar (meetings, lunch, focus blocks). "
            "Call before proposing time blocks or meetings to avoid conflicts."
        ),
        "parameters": {
            "type": "object",
            "properties": {"date": {"type": "string", "description": "Local date YYYY-MM-DD"}},
            "required": ["date"],
        },
    },
    {
        "type": "function",
        "name": "add_calendar_block",
        "description": (
            "Block focus time for a subtask. Use non-overlapping HH:MM; retry if overlap error."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string"},
                "start_time": {"type": "string", "description": "HH:MM 24h"},
                "end_time": {"type": "string", "description": "HH:MM 24h"},
                "title": {"type": "string"},
                "notes": {"type": "string"},
            },
            "required": ["date", "start_time", "end_time", "title"],
        },
    },
    {
        "type": "function",
        "name": "schedule_team_call",
        "description": (
            "Schedule a sync with a team. Must include a concrete agenda: goals, decisions, timeboxes."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string"},
                "start_time": {"type": "string"},
                "end_time": {"type": "string"},
                "team": {"type": "string"},
                "meeting_title": {"type": "string"},
                "agenda": {"type": "string"},
                "related_issue_key": {"type": "string"},
            },
            "required": ["date", "start_time", "end_time", "team", "meeting_title", "agenda"],
        },
    },
]


def get_tool_schemas(*, include_web_search: bool = False) -> list[dict[str, Any]]:
    schemas: list[dict[str, Any]] = list(_FUNCTION_TOOL_SCHEMAS)
    if include_web_search:
        schemas.append({"type": "web_search"})
    return schemas


def execute_tool(name: str, args: Any) -> Any:
    if isinstance(args, str):
        args = json.loads(args)
    if name == "fetch_jira_issue":
        return fetch_jira_issue(**args)
    if name == "list_day_calendar":
        return list_day_calendar(**args)
    if name == "add_calendar_block":
        return add_calendar_block(**args)
    if name == "schedule_team_call":
        return schedule_team_call(**args)
    return {"error": "Unknown tool"}


### Responses API agent loop

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
RESPONSES_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")


def _output_to_string(result: Any) -> str:
    if isinstance(result, (dict, list)):
        return json.dumps(result)
    return str(result)


def _content_to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, list):
        return " ".join(
            str(c.get("text", c.get("content", c))) if isinstance(c, dict) else str(c) for c in content
        )
    return str(content)


def _normalize_user_message(user_input: Any) -> str:
    if user_input is None:
        return ""
    if isinstance(user_input, dict):
        return _content_to_text(user_input.get("content")).strip()
    return str(user_input).strip()


def _extract_text_from_response(response: Any) -> str:
    text_parts: list[str] = []
    for item in getattr(response, "output", None) or []:
        if getattr(item, "type", None) == "message":
            for block in getattr(item, "content", None) or []:
                t = getattr(block, "text", None)
                if t:
                    text_parts.append(t if isinstance(t, str) else "".join(t))
    return "".join(text_parts).strip()


def _build_input_from_history(history: list | None, user_input: Any) -> list:
    user_text = _normalize_user_message(user_input)
    input_list: list = []
    if history:
        for turn in history:
            if not isinstance(turn, dict):
                continue
            role = turn.get("role")
            content = turn.get("content")
            if role not in ("user", "assistant") or content is None:
                continue
            text = _content_to_text(content).strip()
            if not text:
                continue
            input_list.append({"role": role, "content": text})

    if user_text:
        if input_list and input_list[-1].get("role") == "user":
            last = str(input_list[-1].get("content", "")).strip()
            if last == user_text:
                return input_list
        input_list.append({"role": "user", "content": user_text})

    return input_list


def run_agent(
    user_input: Any,
    history: list | None = None,
    *,
    include_web_search: bool = False,
    instructions_suffix: str = "",
) -> str:
    input_list = _build_input_from_history(history, user_input)
    tools = get_tool_schemas(include_web_search=include_web_search)
    instructions = SYSTEM_PROMPT + (instructions_suffix or "")
    max_rounds = 40
    for _ in range(max_rounds):
        response = client.responses.create(
            model=RESPONSES_MODEL,
            instructions=instructions,
            input=input_list,
            tools=tools,
        )
        input_list += list(response.output)

        function_calls = [
            item
            for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        if not function_calls:
            status = getattr(response, "status", None) or ""
            if str(status).lower() == "incomplete":
                continue
            final = (getattr(response, "output_text", None) or "").strip()
            if not final:
                final = _extract_text_from_response(response)
            return apply_guardrails(final or "(No text response from model.)")

        for item in function_calls:
            try:
                result = execute_tool(item.name, item.arguments)
            except Exception as e:
                result = {"error": str(e)}
            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": _output_to_string(result),
                }
            )

    return apply_guardrails(
        "Stopped after too many model rounds. Try a shorter question or check API limits."
    )


### Agents SDK: planner + parallel web search

In [ ]:
_MAX_SEARCHES = max(1, min(8, int(os.getenv("JIRA_BUDDY_AGENTS_MAX_SEARCHES", "4"))))
_PLANNER_MODEL = os.getenv("JIRA_BUDDY_AGENTS_PLANNER_MODEL", "gpt-4o-mini")
_SEARCH_MODEL = os.getenv("JIRA_BUDDY_AGENTS_SEARCH_MODEL", "gpt-4o-mini")


class DevSearchItem(BaseModel):
    reason: str = Field(
        description="Why this search matters (tests, security, API semantics, ops, version-specific behavior)."
    )
    query: str = Field(description="Short, keyword-style web search query.")


class DevSearchPlan(BaseModel):
    searches: list[DevSearchItem] = Field(
        description="Ordered web searches to ground estimates, edge cases, and test ideas.",
    )


def _planner_instructions(n: int) -> str:
    return f"""You plan **web research** for a **junior developer** working a Jira-style task.

Given the task (summary, tech stack hints, acceptance criteria if any), propose **up to {n}**
focused searches that will help with:
- Correct implementation patterns and pitfalls
- Testing strategy (unit/integration/e2e) and tooling
- Security, reliability, idempotency, or data-handling concerns when relevant
- Current docs or migration notes when the stack is named (framework, cloud, DB)

Rules:
- Prefer **specific** queries (library names, error concepts, release years when freshness matters).
- **At most {n}** items in `searches`; use fewer only if the task is very narrow.
- Do **not** search for "the user's calendar" or "Jira ticket text" — another system handles that.
"""


def _search_instructions() -> str:
    return """You are a **web research subagent** for a software task.

You receive one `Search term` and `Reason`. Use the web search tool to gather **current**,
**source-backed** notes (official docs, reputable posts, CVE/advisory pages when relevant).

Reply with **2–4 short paragraphs** (under ~350 words): facts, links implied by citations the tool
returns, and **actionable implications** for a junior dev (what to watch for in code/tests).
No meta-commentary about being an AI."""


planner_agent = Agent(
    name="DevResearchPlanner",
    instructions=_planner_instructions(_MAX_SEARCHES),
    model=_PLANNER_MODEL,
    output_type=DevSearchPlan,
    model_settings=ModelSettings(max_tokens=1200),
)

search_agent = Agent(
    name="DevWebSearch",
    instructions=_search_instructions(),
    tools=[WebSearchTool(search_context_size="low")],
    model=_SEARCH_MODEL,
    model_settings=ModelSettings(
        max_tokens=2000,
        tool_choice="required",
    ),
)


async def _run_agents_research(user_text: str) -> str:
    with trace("Jira Buddy — planner + search subagents"):
        plan_result = await Runner.run(
            planner_agent,
            f"Plan web research for this development task:\n\n{user_text}",
        )
        plan = plan_result.final_output
        if not isinstance(plan, DevSearchPlan):
            return f"(Planner returned unexpected output: {plan!r})"

        items = plan.searches[:_MAX_SEARCHES]
        if not items:
            return "(Planner returned no searches.)"

        async def one(item: DevSearchItem):
            prompt = f"Search term: {item.query}\nReason: {item.reason}"
            res = await Runner.run(search_agent, prompt)
            return res.final_output

        parts = await asyncio.gather(*[one(it) for it in items])
        return "\n\n---\n\n".join(str(p) for p in parts)


def run_jira_buddy(user_input: Any, history: list | None = None) -> str:
    user_text = _normalize_user_message(user_input)
    if not user_text.strip():
        return "Send a task description or Jira issue details to get started."

    try:
        research = _run_coro_sync(lambda: _run_agents_research(user_text))
    except Exception as e:
        return f"Agents SDK research failed: {e!s}\nCheck OPENAI_API_KEY and network, then retry."

    augmented = (
        f"{user_text}\n\n---\n### Web research (planner + search subagents)\n\n{research}"
    )
    return run_agent(augmented, history, include_web_search=False)


### Gradio
Launch the chat UI (inline in Jupyter when supported).

In [ ]:
def chat(message: Any, history: list | None) -> str:
    try:
        return run_jira_buddy(message, history)
    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Jira Dev Buddy",
    description=(
        "Planner + parallel **WebSearchTool** agents, then Jira/calendar tools via **Responses API**. "
        "[Traces](https://platform.openai.com/traces)"
    ),
)

demo.launch(inline=True)
